In [ ]:
import numpy as np
import pandas as pd
from scipy.io import mmread
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import coo_matrix
from datetime import timedelta, datetime
import warnings
warnings.filterwarnings('ignore')

k=0
metric='cosine'
###############################################################
# 1)  SNN Jaccard
###############################################################
def build_snn_jaccard(X, k=k, metric=metric, include_self=False):
    nbrs = NearestNeighbors(n_neighbors=k, metric=metric, n_jobs=-1).fit(X)
    distances, indices = nbrs.kneighbors(X, return_distance=True)
    n = X.shape[0]

    neigh_sets = [set(row) for row in indices]

    rows, cols, vals = [], [], []

    for i in range(n):
        Ni = neigh_sets[i].copy()
        if include_self:
            Ni.add(i)

        for j in indices[i]:
            Nj = neigh_sets[j].copy()
            if include_self:
                Nj.add(j)

            inter = len(Ni & Nj)
            union = len(Ni | Nj)
            if union > 0:
                jacc = inter / union
                if jacc > 0:
                    rows.append(i)
                    cols.append(j)
                    vals.append(jacc)

    M = coo_matrix((vals, (rows, cols)), shape=(n, n)).tocsr()
    M_sym = M.maximum(M.T)

    return M_sym
# ===== LOAD DATA =====
X_rna = mmread('BMMCData/Gene_Cell_atac.mtx').tocsr()
genes = pd.read_csv('BMMCData/Gene_names_atac.tsv', header=None)[0].values
cells = pd.read_csv('BMMCData/Cell_names_atac.tsv', header=None)[0].values
cell_type = pd.read_csv('BMMCData/Cell_type_atac.tsv', header=None)[0].values
X_atac = mmread('BMMCData/Peak_Cell.mtx').tocsr()#.transpose()
peaks = pd.read_csv('BMMCData/Peak_names.tsv', header=None)[0].values

now = datetime.now()
print(f"⏰ Start Mutual KNN -ATAC: {now.strftime('%H:%M:%S')}")
print(f"ATAC data shape: {X_atac.shape}")

print(f"RNA data shape: {X_rna.shape}")


from scipy.sparse import csr_matrix, issparse
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
now = datetime.now()
print(f"⏰ Start Mutual KNN -RNA: {now.strftime('%H:%M:%S')}")


# ===== PREPROCESSING =====
# Step 1: Filter genes (keep genes expressed in at least 3 cells)
gene_counts = np.array((X_rna > 0).sum(axis=0)).flatten()
genes_keep = gene_counts >= 3
X_rna = X_rna[:, genes_keep]
genes = genes[genes_keep]
print(f"After filtering: {X_rna.shape}")

# Step 2: Normalize by total counts per cell (library size normalization)
cell_sums = np.array(X_rna.sum(axis=1)).flatten()
X_rna_norm = X_rna.multiply(1 / cell_sums[:, np.newaxis]).multiply(10000)

# Step 3: Log transformation: log1p(x) = log(1 + x)
X_rna_log = X_rna_norm.copy()
X_rna_log.data = np.log1p(X_rna_log.data)
# Step 4: Feature selection - select highly variable genes

import scanpy as sc

#n_top_genes = min(2000, X_rna_log.shape[1])
n_top_genes = min(3000, X_rna_log.shape[1])

adata_temp = sc.AnnData(X_rna_log.tocsr())

sc.pp.highly_variable_genes(
    adata_temp,
    n_top_genes=n_top_genes,
    flavor= 'seurat'
)

hvg_mask = adata_temp.var['highly_variable']

if issparse(X_rna_log):
    X_dense = X_rna_log.toarray()
else:
    X_dense = X_rna_log

X_hvg = X_dense[:, hvg_mask]

print(f"Selected {X_hvg.shape[1]} highly variable genes (using 'seurat' method)")

# Step 5: Standardize (z-score normalization)
scaler = StandardScaler(with_mean=True, with_std=True)
X_scaled = scaler.fit_transform(X_hvg)

# Step 6: PCA for dimensionality reduction
n_pcs = 30
pca = PCA(n_components=n_pcs, random_state=42)
X_pca = pca.fit_transform(X_scaled)
print(f"PCA variance explained: {pca.explained_variance_ratio_.sum():.3f}")
adata_rna = sc.AnnData(X_rna_log.tocsr())
adata_rna.obs['cell_type'] = cell_type
adata_rna.obsm['X_pca'] = X_pca

# SNN
from scipy.sparse import coo_matrix, csr_matrix
import scanpy as sc
from scipy.sparse.csgraph import connected_components

k_rna=40
metric='cosine'
adata_rna = sc.AnnData(X_rna_log.tocsr())
adata_rna.obs['cell_type'] = cell_type
adata_rna.obsm['X_pca'] = X_pca
#################
#SNN
###############################################################
snn_graph = build_snn_jaccard(X_pca, k=k_rna, metric=metric, include_self=False)
graph_rna_scanpy = snn_graph
print("SNN nnz:", snn_graph.nnz)


from scipy.sparse import load_npz
from node2vec import Node2Vec
import inspect
import warnings
import networkx as nx
print(nx.__version__)
warnings.filterwarnings('ignore')
from datetime import timedelta, datetime
now = datetime.now()
print(f"⏰ Start Mutual KNN -ATAC: {now.strftime('%H:%M:%S')}")

print("="*70)
print("RNA GRAPH NODE2VEC EMBEDDING")
print("="*70)

# ===== LOAD RNA mkNN GRAPH =====
print("\n[1/6] Loading RNA mkNN graph...")
try:
    mknn_graph_rna = graph_rna_scanpy 
    print(f"✓ RNA graph loaded: {mknn_graph_rna.shape}")
    print(f"  - Nodes (cells): {mknn_graph_rna.shape[0]}")
    print(f"  - Edges: {mknn_graph_rna.nnz // 2}")
    print(f"  - Density: {mknn_graph_rna.nnz / (mknn_graph_rna.shape[0]**2):.6f}")
except Exception as e:
    print(f"✗ Error loading graph: {e}")
    raise

# ===== CONVERT TO NETWORKX GRAPH =====
print("\n[2/6] Converting to NetworkX graph...")
G_rna = nx.from_scipy_sparse_array(mknn_graph_rna)

# Graph statistics
print(f"✓ NetworkX graph created")
print(f"  - Nodes: {G_rna.number_of_nodes()}")
print(f"  - Edges: {G_rna.number_of_edges()}")
print(f"  - Connected: {nx.is_connected(G_rna)}")
if not nx.is_connected(G_rna):
    components = list(nx.connected_components(G_rna))
    print(f"  - Number of components: {len(components)}")
    print(f"  - Largest component size: {len(max(components, key=len))}")

# ===== NODE2VEC PARAMETERS =====
print("\n[3/6] Setting up Node2Vec parameters...")
config = {
    'dimensions': 50, #128,
    'walk_length': 80,
    'num_walks': 10,
    'p': 1.0,
    'q': 1.0,
    'workers': 1,  # FIX: Set to 1 to avoid multiprocessing issues
    'window': 10,
    'min_count': 1,
    'batch_words': 4,
    'epochs': 20,
    'seed': 42
}

print("Configuration:")
for key, value in config.items():
    print(f"  - {key}: {value}")

# ===== GENERATE RANDOM WALKS =====
print("\n[4/6] Generating random walks...")
print("  (This may take a few minutes depending on graph size)")

try:
    node2vec_rna = Node2Vec(
        G_rna,
        dimensions=config['dimensions'],
        walk_length=config['walk_length'],
        num_walks=config['num_walks'],
        p=config['p'],
        q=config['q'],
        workers=config['workers'],  # Single worker to avoid pickling errors
        quiet=False,
        seed=config['seed']
    )
    print(f"✓ Generated {len(node2vec_rna.walks)} walks")
    print(f"  - Total steps: {len(node2vec_rna.walks) * config['walk_length']}")
    
except Exception as e:
    print(f"✗ Error during walk generation: {e}")
    raise

# ===== TRAIN NODE2VEC MODEL =====
print("\n[5/6] Training Node2Vec model...")
print(f"  Training for {config['epochs']} epochs...")

try:
    model_rna = node2vec_rna.fit(
        window=config['window'],
        min_count=config['min_count'],
        batch_words=config['batch_words'],
        epochs=config['epochs'],
        seed=config['seed']
    )
    print("✓ Model training complete")
    
except Exception as e:
    print(f"✗ Error during training: {e}")
    raise

# ===== EXTRACT EMBEDDINGS =====
print("\n[6/6] Extracting embeddings...")

n_nodes = mknn_graph_rna.shape[0]
embeddings_rna = np.zeros((n_nodes, config['dimensions']))

for i in range(n_nodes):
    try:
        embeddings_rna[i] = model_rna.wv[str(i)]
    except KeyError:
        print(f"  Warning: Node {i} not found in model, using zeros")
        embeddings_rna[i] = np.zeros(config['dimensions'])

print(f"✓ Embeddings extracted: {embeddings_rna.shape}")
print(f"  - Mean: {embeddings_rna.mean():.4f}")
print(f"  - Std: {embeddings_rna.std():.4f}")
print(f"  - Min: {embeddings_rna.min():.4f}")
print(f"  - Max: {embeddings_rna.max():.4f}")

# ===== VALIDATE EMBEDDINGS =====
print("\n[Validation] Checking embedding quality...")
# Check for NaN or Inf values
has_nan = np.isnan(embeddings_rna).any()
has_inf = np.isinf(embeddings_rna).any()
print(f"  - Contains NaN: {has_nan}")
print(f"  - Contains Inf: {has_inf}")

# Check embedding diversity
embedding_norms = np.linalg.norm(embeddings_rna, axis=1)
print(f"  - Embedding norms - Mean: {embedding_norms.mean():.4f}, Std: {embedding_norms.std():.4f}")

# Check for duplicate embeddings
unique_embeddings = np.unique(embeddings_rna, axis=0)
print(f"  - Unique embeddings: {len(unique_embeddings)} / {n_nodes}")

# ===== SAVE RESULTS =====
print("\n[Saving] Writing results to disk...")
np.save('node2vec_embeddings_rna.npy', embeddings_rna)



# --- File paths ---
data_path = "node2vec_embeddings_rna.npy"
label_path = "BMMCData/Cell_type_atac.tsv"
output_path = "combined_embeddings_RNA.csv"

# --- Load data ---
data = np.load(data_path)  # shape: (n_samples, n_features)
labels = pd.read_csv(label_path, sep='\t', header=None)  # adjust header if needed

# --- Check shapes ---
print("Data shape:", data.shape)
print("Labels shape:", labels.shape)

# Ensure label length matches number of rows
if len(labels) != data.shape[0]:
    raise ValueError("Number of labels does not match number of rows in data!")

# --- Combine data and labels ---
combined = np.column_stack((data, labels.values))  # add labels as last column

# --- Save as CSV ---
# Create a DataFrame for better readability (optional)
columns = [f"feature_{i+1}" for i in range(data.shape[1])] + ["label"]
df = pd.DataFrame(combined, columns=columns)

df.to_csv(output_path, index=False)

print(f"Saved combined data to {output_path}")
from sklearn.decomposition import TruncatedSVD
# ===== PREPROCESSING =====
# Step 1: Filter peaks (keep peaks accessible in at least 5 cells)
peak_counts = np.array((X_atac > 0).sum(axis=0)).flatten()
peaks_keep = peak_counts >= 5
X_atac = X_atac[:, peaks_keep]
peaks = peaks[peaks_keep]
print(f"After filtering: {X_atac.shape}")

# Step 2: Binarize (ATAC is often binary: peak present or not)
X_atac_binary = X_atac.copy()
X_atac_binary.data = np.ones_like(X_atac_binary.data)

# Step 3: TF-IDF normalization (common for ATAC-seq)
# TF (Term Frequency): normalize by total accessibility per cell
cell_sums = np.array(X_atac_binary.sum(axis=1)).flatten()
tf = X_atac_binary.multiply(1 / (cell_sums[:, np.newaxis] + 1e-10))

# IDF (Inverse Document Frequency): weight by peak rarity
n_cells = X_atac_binary.shape[0]
peak_frequencies = np.array((X_atac_binary > 0).sum(axis=0)).flatten()
idf = np.log(1 + n_cells / (peak_frequencies + 1))

# Apply TF-IDF
X_tfidf = tf.multiply(idf)
print("TF-IDF normalization completed")

if issparse(X_tfidf):
    # Compute variance on sparse matrix
    mean = np.array(X_tfidf.mean(axis=0)).ravel()
    mean_sq = np.array(X_tfidf.multiply(X_tfidf).mean(axis=0)).ravel()
    peak_vars = mean_sq - mean**2
else:
    peak_vars = np.var(X_tfidf, axis=0)


#n_top_peaks = min(25000, X_tfidf.shape[1])
n_top_peaks = min(50000, X_tfidf.shape[1])

top_peaks_idx = np.argsort(peak_vars)[-n_top_peaks:]

from scipy.sparse import csr_matrix

X_tfidf = csr_matrix(X_tfidf)  
X_variable_sparse = X_tfidf[:, top_peaks_idx]
print(f"Selected {n_top_peaks} variable peaks")

# Step 6: LSI/SVD for dimensionality reduction (PCA alternative for sparse data)
n_components = 150

svd = TruncatedSVD(n_components=n_components, random_state=42)
#X_lsi = svd.fit_transform(X_scaled)
X_lsi = svd.fit_transform(X_variable_sparse)
X_lsi= X_lsi[:, 1:]################################################################################################################################################
print(f"LSI variance explained: {svd.explained_variance_ratio_.sum():.3f}")

# SNN
k=k_atac=60
metric='cosine'
adata_atac = sc.AnnData(X_variable_sparse)
adata_atac.obs['cell_type' ] = cell_type#. values
adata_atac.obsm['X_lsi'] = X_lsi
###############################################################
# 2) SNN
###############################################################
snn_graph2 = build_snn_jaccard(X_lsi, k=k_atac, metric=metric, include_self=False)
graph_atac_scanpy = snn_graph2
print("SNN nnz:", snn_graph2.nnz)


print("="*70)
print("ATAC GRAPH NODE2VEC EMBEDDING")
print("="*70)
now = datetime.now()
print(f"⏰ start: {now.strftime('%H:%M:%S')}")
# ===== LOAD ATAC mkNN GRAPH =====
print("\n[1/6] Loading ATAC mkNN graph...")
try:
    mknn_graph_atac = graph_atac_scanpy # load_npz('mknn_graph_atac.npz')
    print(f"✓ ATAC graph loaded: {mknn_graph_atac.shape}")
    print(f"  - Nodes (cells): {mknn_graph_atac.shape[0]}")
    print(f"  - Edges: {mknn_graph_atac.nnz // 2}")
    print(f"  - Density: {mknn_graph_atac.nnz / (mknn_graph_atac.shape[0]**2):.6f}")
except Exception as e:
    print(f"✗ Error loading graph: {e}")
    raise

# ===== CONVERT TO NETWORKX GRAPH =====
print("\n[2/6] Converting to NetworkX graph...")
G_atac = nx.from_scipy_sparse_array(mknn_graph_atac)

# Graph statistics
print(f"✓ NetworkX graph created")
print(f"  - Nodes: {G_atac.number_of_nodes()}")
print(f"  - Edges: {G_atac.number_of_edges()}")
print(f"  - Connected: {nx.is_connected(G_atac)}")
if not nx.is_connected(G_atac):
    components = list(nx.connected_components(G_atac))
    print(f"  - Number of components: {len(components)}")
    print(f"  - Largest component size: {len(max(components, key=len))}")

# ===== NODE2VEC PARAMETERS =====
print("\n[3/6] Setting up Node2Vec parameters...")
config = {
    'dimensions': 50,
    'walk_length': 80,
    'num_walks': 10,
    'p': 1.0,
    'q': 1.0,
    'workers': 1,  # FIX: Set to 1 to avoid multiprocessing issues
    'window': 10,
    'min_count': 1,
    'batch_words': 4,
    'epochs': 20,
    'seed': 42
}

print("Configuration:")
for key, value in config.items():
    print(f"  - {key}: {value}")

# ===== GENERATE RANDOM WALKS =====
print("\n[4/6] Generating random walks...")
print("  (This may take a few minutes depending on graph size)")

try:
    node2vec_atac = Node2Vec(
        G_atac,
        dimensions=config['dimensions'],
        walk_length=config['walk_length'],
        num_walks=config['num_walks'],
        p=config['p'],
        q=config['q'],
        workers=config['workers'],  # Single worker to avoid pickling errors
        quiet=False,
        seed=config['seed']
    )
    print(f"✓ Generated {len(node2vec_atac.walks)} walks")
    print(f"  - Total steps: {len(node2vec_atac.walks) * config['walk_length']}")
    
except Exception as e:
    print(f"✗ Error during walk generation: {e}")
    raise

# ===== TRAIN NODE2VEC MODEL =====
print("\n[5/6] Training Node2Vec model...")
print(f"  Training for {config['epochs']} epochs...")

try:
    model_atac = node2vec_atac.fit(
        window=config['window'],
        min_count=config['min_count'],
        batch_words=config['batch_words'],
        epochs=config['epochs'],
        seed=config['seed']
    )
    print("✓ Model training complete")
    
except Exception as e:
    print(f"✗ Error during training: {e}")
    raise

# ===== EXTRACT EMBEDDINGS =====
print("\n[6/6] Extracting embeddings...")

n_nodes = mknn_graph_atac.shape[0]
embeddings_atac = np.zeros((n_nodes, config['dimensions']))

for i in range(n_nodes):
    try:
        embeddings_atac[i] = model_atac.wv[str(i)]
    except KeyError:
        print(f"  Warning: Node {i} not found in model, using zeros")
        embeddings_atac[i] = np.zeros(config['dimensions'])

print(f"✓ Embeddings extracted: {embeddings_atac.shape}")
print(f"  - Mean: {embeddings_atac.mean():.4f}")
print(f"  - Std: {embeddings_atac.std():.4f}")
print(f"  - Min: {embeddings_atac.min():.4f}")
print(f"  - Max: {embeddings_atac.max():.4f}")

# ===== VALIDATE EMBEDDINGS =====
print("\n[Validation] Checking embedding quality...")
# Check for NaN or Inf values
has_nan = np.isnan(embeddings_atac).any()
has_inf = np.isinf(embeddings_atac).any()
print(f"  - Contains NaN: {has_nan}")
print(f"  - Contains Inf: {has_inf}")

# Check embedding diversity
embedding_norms = np.linalg.norm(embeddings_atac, axis=1)
print(f"  - Embedding norms - Mean: {embedding_norms.mean():.4f}, Std: {embedding_norms.std():.4f}")

# Check for duplicate embeddings
unique_embeddings = np.unique(embeddings_atac, axis=0)
print(f"  - Unique embeddings: {len(unique_embeddings)} / {n_nodes}")

# ===== SAVE RESULTS =====
print("\n[Saving] Writing results to disk...")
np.save('node2vec_embeddings_atac.npy', embeddings_atac)
print("✓ Embeddings saved: 'node2vec_embeddings_atac.npy'")


input_path = "node2vec_embeddings_atac.npy"
output_path = "node2vec_embeddings_atac.csv"
data = np.load(input_path)
df = pd.DataFrame(data, columns=[f"feature_{i+1}" for i in range(data.shape[1])])
df.to_csv(output_path, index=False)
print(f"Saved CSV file to {output_path}")


######################################################################
######################################################################
######################################################################
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
from sklearn.preprocessing import normalize, LabelEncoder
from sklearn.decomposition import TruncatedSVD

# ============================================
# READ CSV DATA
# ============================================
print("Reading CSV files...")
df_rna = pd.read_csv('combined_embeddings_RNA.csv')
df_atac = pd.read_csv('node2vec_embeddings_atac.csv')

# RNA: Separate features and labels
X_rna = df_rna.iloc[:, :-1].values  # All columns except last
y_labels = df_rna.iloc[:, -1].values  # Last column (labels)

# ATAC: All columns are features
X_atac = df_atac.values

print(f"RNA data shape: {X_rna.shape}")
print(f"ATAC data shape: {X_atac.shape}")
print(f"Number of samples: {len(X_rna)}")
print(f"Number of RNA features: {X_rna.shape[1]}")
print(f"Number of ATAC features: {X_atac.shape[1]}")
print(f"Number of classes: {len(np.unique(y_labels))}")
print(f"Class distribution: {np.unique(y_labels, return_counts=True)}")

# ============================================
# USE ALL DATA FOR TRAINING AND EMBEDDING
# ============================================
print("\n" + "="*60)
print("USING ALL DATA FOR TRAINING AND EMBEDDING")
print("="*60)

# Use ALL samples for both training and embedding
X_rna_train = X_rna  # ALL samples
X_atac_train = X_atac  # ALL samples

X_rna_test = X_rna  # Same as training (all samples)
X_atac_test = X_atac  # Same as training (all samples)
data_labels = y_labels  # All labels
cell_types=y_labels
print(f"\nTotal samples used: {len(X_rna)}")
print(f"Training samples: {len(X_rna_train)} (100% of data)")
print(f"Embedding samples: {len(X_rna_test)} (100% of data)")

# Random Forest

# ============================================
# TRAIN RANDOM FOREST REGRESSORS (RNA → ATAC)
# WITH IMPROVED PARAMETERS
# ============================================
print("\n" + "="*60)
print("TRAINING IMPROVED RANDOM FOREST REGRESSORS (RNA → ATAC)")
print("="*60)

# IMPROVED PARAMETERS
nt = 200  # Number of trees per forest
n_atac_features = X_atac.shape[1]
total_rfs = n_atac_features  # One RF per ATAC feature

rf_models = []

print(f"\nTraining RNA → ATAC regression models with improved parameters:")
print(f"  - Number of trees per forest: {nt}")
print(f"  - Min samples per leaf: 3")
print(f"  - Max features: 'sqrt' (better feature selection)")
print(f"  - Max depth: 20 (controlled depth)")
print(f"Training {n_atac_features} Random Forests...")
print("-" * 60)

for target_idx in range(n_atac_features):
    print(f"Progress: [{target_idx+1}/{total_rfs}] Training RF for ATAC feature {target_idx+1}...", end='')
    
    # Input: ALL RNA features, Output: single ATAC feature
    X_train_input = X_rna_train  # All RNA features as input
    y_train_target = X_atac_train[:, target_idx]  # Single ATAC feature as target
    
    # IMPROVED RANDOM FOREST PARAMETERS
    rf = RandomForestRegressor(
        n_estimators=nt,
        min_samples_leaf=3,
        max_depth=20,
        max_features='sqrt',
        min_samples_split=5,
        bootstrap=True,
        random_state=42,
        n_jobs=-1
    )
    
    # Input: all RNA features, Output: single ATAC feature
    rf.fit(X_train_input, y_train_target)
    rf_models.append(rf)
    
    # Calculate R² score on training data
    r2_score_2 = rf.score(X_train_input, y_train_target)
    print(f" ✓ Complete (R² score: {r2_score_2:.3f})")

print("\n" + "="*60)
print(f"All {total_rfs} Random Forests trained successfully!")
print("="*60)

# cell-leaf

# ============================================
# EXTRACT LEAF NODES FROM ALL RFs
# ============================================
print("\nExtracting leaf nodes from all Random Forests...")

all_leaf_indices = []

# Extract from RNA → ATAC models (using RNA data)
for target_idx in range(n_atac_features):
    # Use RNA data as input to get leaf indices
    leaf_indices = rf_models[target_idx].apply(X_rna_test)
    all_leaf_indices.append(leaf_indices)

# Concatenate all leaf indices
# Shape: (n_samples, total_rfs * nt)
all_leaf_indices = np.hstack(all_leaf_indices)
print(f"Combined leaf indices shape: {all_leaf_indices.shape}")

# Print matrix information
print(f"\nLeaf Index Matrix Information:")
print(f"Number of samples: {all_leaf_indices.shape[0]}")
print(f"Number of features (trees): {all_leaf_indices.shape[1]}")
print(f"Memory size: {all_leaf_indices.nbytes / 1024 / 1024:.2f} MB")
print(f"Data type: {all_leaf_indices.dtype}")

# Embeding

# ============================================
# SVD-BASED EMBEDDING
# ============================================
print("\n" + "="*60)
print("APPLYING SVD-BASED EMBEDDING")
print("="*60)

import time
start_time = time.time()

# Normalize leaf indices for better numerical stability
print("Normalizing leaf index matrix...")
all_leaf_indices_normalized = normalize(all_leaf_indices.astype(float), norm='l2')

# Apply Truncated SVD for dimensionality reduction
print("Applying Truncated SVD...")
embedding_dim = 300  # Optimal dimension
svd = TruncatedSVD(n_components=embedding_dim, random_state=42)
rf_embeddings = svd.fit_transform(all_leaf_indices_normalized)

# Normalize embeddings
rf_embeddings = normalize(rf_embeddings, norm='l2')
pd.DataFrame(rf_embeddings).to_csv("our_integrated.csv", index=False)
pd.DataFrame({'cell_type': cell_types}).to_csv("our_celltypes.csv", index=False)
embedding_time = time.time() - start_time
print(f"✓ SVD embedding completed in {embedding_time:.2f} seconds")
print(f"Embedding shape: {rf_embeddings.shape}")
print(f"Explained variance ratio: {svd.explained_variance_ratio_.sum():.4f}")